# Predicción de Tiempos de Entrega — Olist Marketplace

Objetivo: predecir `target_days` (días entre compra y entrega). Flujo del notebook: carga → preparación y features → limpieza → modelado y comparativa → evaluación → importancia → diagnóstico y conclusiones.


## 1. Carga de datos

Se cargan los 6 CSVs de Olist (órdenes, items, productos, clientes, vendedores, geolocalización).


In [ ]:
import numpy as np
import pandas as pd

path = "datasets"

# Carga de datasets
orders = pd.read_csv(f'{path}/olist_orders_dataset.csv')
items = pd.read_csv(f'{path}/olist_order_items_dataset.csv')
products = pd.read_csv(f'{path}/olist_products_dataset.csv')
customers = pd.read_csv(f'{path}/olist_customers_dataset.csv')
sellers = pd.read_csv(f'{path}/olist_sellers_dataset.csv')
geo = pd.read_csv(f'{path}/olist_geolocation_dataset.csv')

### 1.1 Función auxiliar de distancia (Haversine)

Define la distancia en km entre vendedor y cliente, base de la feature `distance_km`.


In [ ]:
from sklearn.metrics.pairwise import haversine_distances


def haversine(lat1, lon1, lat2, lon2):
    # El radio de la Tierra en kilómetros
    r = 6371
    
    # Preparar los puntos en radianes para sklearn (espera pares [lat, lon])
    point1 = np.radians([[lat1, lon1]])
    point2 = np.radians([[lat2, lon2]])
    
    # Calcular la distancia
    result = haversine_distances(point1, point2)
        # Multiplicar por el radio para obtener el valor en km
    return result[0][0] * r

## 2. Preparación de datos y feature engineering

Construye el dataset unificado en 5 pasos: geolocalización/volumen (2.1), filtro single-seller (2.2), agregación por orden (2.3), merges + distancia (2.4), fechas y `target_days` (2.5).


### 2.1 Geolocalización y volumen

Promedia coordenadas por ZIP y calcula el volumen en cm³ de cada producto.


In [ ]:
geo_coords = geo.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat' : 'mean',
    'geolocation_lng' : 'mean'
}).reset_index()

products['product_volume_cm3'] = products['product_length_cm'] * products['product_height_cm'] * products['product_width_cm']

### 2.2 Filtro single-seller

Descarta órdenes con múltiples vendedores (~1.3%) y filtra `items` a órdenes válidas.


In [ ]:
sellers_per_order = items.groupby('order_id')['seller_id'].nunique()
multi_seller_count = (sellers_per_order > 1).sum()

print(f"Órdenes con 1 solo vendedor: {(sellers_per_order == 1).sum()}")
print(f"Órdenes con múltiples vendedores: {multi_seller_count}")
print(f"Porcentaje a descartar: {multi_seller_count / len(sellers_per_order) * 100:.2f}%")

valid_order_ids = sellers_per_order[sellers_per_order == 1].index

items_single_seller = items[items['order_id'].isin(valid_order_ids)]

### 2.3 Agregación de items por orden

Une peso/volumen a `items` y agrega por orden: peso, volumen, flete y nº items.


In [ ]:
# Agregar datos de productos a items
items_single_seller = items_single_seller.merge(products[['product_id', 'product_weight_g', 'product_volume_cm3']], on='product_id')

orders_items_agg = items_single_seller.groupby(['order_id', 'seller_id']).agg(
    total_weight_g=('product_weight_g', 'sum'),
    total_volume_cm3=('product_volume_cm3', 'sum'),
    total_freight=('freight_value', 'sum'),
    n_items=('order_item_id', 'count')    
).reset_index()

### 2.4 Merges y distancia

Une órdenes/clientes/vendedores/geolocalización; calcula `distance_km` y `is_same_state`.


In [ ]:
df = orders.merge(orders_items_agg, on='order_id')
df = df.merge(customers[['customer_id', 'customer_zip_code_prefix', 'customer_state']], on='customer_id')
df = df.merge(sellers[['seller_id', 'seller_zip_code_prefix', 'seller_state']], on='seller_id')

df = df.merge(geo_coords, left_on='customer_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='left')\
    .rename(columns={'geolocation_lat' : 'cust_lat', 'geolocation_lng' : 'cust_lng'})\
    .drop(columns='geolocation_zip_code_prefix')

df = df.merge(geo_coords, left_on='seller_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='left')\
    .rename(columns={'geolocation_lat' : 'sel_lat', 'geolocation_lng' : 'sel_lng'})\
    .drop(columns='geolocation_zip_code_prefix')

# Manejo de nulos en coordenadas antes de aplicar Haversine
df = df.dropna(subset=['cust_lat', 'cust_lng', 'sel_lat', 'sel_lng'])

df['distance_km'] = [haversine(lat1, lon1, lat2, lon2) for lat1, lon1, lat2, lon2 in zip(df['cust_lat'], df['cust_lng'], df['sel_lat'], df['sel_lng'])]

df['is_same_state'] = (df['customer_state'] == df['seller_state']).astype(int)

### 2.5 Fechas, estacionalidad y target

Parsea fechas, filtra entregados válidos y calcula `order_month`, `target_days` y `estimated_days`.


In [ ]:
date_cols = ['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date']

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Filtrar solo pedidos completados y con fechas no nulas
df = df[df['order_status'] == 'delivered'].dropna(subset=['order_purchase_timestamp', 'order_delivered_customer_date'])

df['order_month'] = df['order_purchase_timestamp'].dt.month

# Filtrar solo los casos válidos (entrega >= compra)
df = df[df['order_delivered_customer_date'] >= df['order_purchase_timestamp']].copy()

# Cálculo del tiempo total de entrega en días (variable target)
df['target_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.total_seconds() / 86400

# Cálculo del tiempo estimado de entrega prometido al cliente
df['estimated_days'] = (df['order_estimated_delivery_date'] - df['order_purchase_timestamp']).dt.total_seconds() / 86400

print(f"Dataset unificado {df.shape}")

## 3. Limpieza de outliers

Corta la cola extrema con el cuantil 0.99 de `target_days` (94725 → 93777 registros).


In [ ]:
print(f"Registros antes de limpiar outliers {len(df)}")

# Filtrar pedidos
limit_days = df['target_days'].quantile(0.99)
df_limpio = df[df['target_days'] <= limit_days].copy()

print(f"Registros despues de limpiar {len(df_limpio)}")

# Actualizar el dataframe principal
df = df_limpio

### 3.1 Cálculo de cuantiles P50, P90, P95 y P99 de `target_days`

Calcula los cuantiles 0.50, 0.90, 0.95 y 0.99 de `target_days` post-limpieza. Reporta además el P99 pre-limpieza (46.1 días) usado como corte de outliers en el punto 3.


In [ ]:
qs = [0.5, 0.9, 0.95, 0.99]
print("Cuantiles de target_days (post-limpieza):")
print(df['target_days'].quantile(qs).round(2).to_string())
print(f"\nP99 pre-limpieza (corte de outliers): {limit_days:.1f} dias")
print("\nLectura: 25 dias cae entre P90 (~22.3) y P95 (~27.9), ver 5.1.")

## 4. Modelado y comparativa de modelos

Compara Dummy (mean), Decision Tree, Random Forest e HistGradientBoosting con el mismo pipeline y métricas MAE/RMSE/R2.


In [ ]:
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor

# Preparacion de datos (ordenamiento por fecha de compra)
df = df.sort_values("order_purchase_timestamp").reset_index(drop=True)

features = [
    "distance_km",
    "estimated_days",
    "total_weight_g",
    "total_volume_cm3",
    "total_freight",
    "n_items",
    "order_month",
    "is_same_state",
]
X = df[features]
y = df["target_days"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False)

# order_month numérico: OneHot se descartó porque empeoró MAE en test temporal.
numeric_features = [
    "distance_km",
    "estimated_days",
    "total_weight_g",
    "total_volume_cm3",
    "total_freight",
    "n_items",
]

preprocessor = ColumnTransformer(transformers=[
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("rest", "passthrough", ["order_month", "is_same_state"]),
])

# Modelos a comparar (Dummy como baseline ingenuo: predice la media del train)
modelos = {
    "Dummy (mean)": DummyRegressor(strategy="mean"),
    "Decision Tree": DecisionTreeRegressor(max_depth=10, random_state=42),
    "Random Forest": RandomForestRegressor(
        n_estimators=100, max_depth=10, n_jobs=-1, random_state=42
    ),
    "Gradient Boosting": HistGradientBoostingRegressor(max_iter=100, random_state=42),
}

results = []

print(" === COMPARATIVA DE MODELOS")
for name, regressor in modelos.items():
    # El baseline Dummy predice la media en escala original: no se envuelve en log1p/expm1
    if isinstance(regressor, DummyRegressor):
        pipeline = Pipeline([("pre", preprocessor), ("reg", regressor)])
    else:
        regressor_log = TransformedTargetRegressor(
            regressor=regressor,
            func=np.log1p,
            inverse_func=np.expm1
        )
        pipeline = Pipeline([("pre", preprocessor), ("reg", regressor_log)])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results.append({"Modelo": name, "MAE": mae, "RMSE": rmse, "R2": r2})
    print(f"{name} -> MAE: {mae:.2f} | RMSE: {rmse:.2f} | R2: {r2:.4f}")

results_df = pd.DataFrame(results).sort_values(by='MAE')
print(f"\nEl mejor modelo es: {results_df['Modelo'].values[0]}")

## 5. Evaluación del mejor modelo

Reentrena HistGradientBoosting y lo evalúa con dos gráficos.

### 5.1 Reales vs Predicciones

Días reales vs predichos; la diagonal roja es predicción perfecta.


In [ ]:
# Re-ejecución del mejor modelo para visualización
import matplotlib.pyplot as plt
import seaborn as sns

best_regressor = TransformedTargetRegressor(
    regressor=HistGradientBoostingRegressor(max_iter=100, random_state=42),
    func=np.log1p,
    inverse_func=np.expm1
)
best_model = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', best_regressor)])
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

plot_best_model = pd.DataFrame({"real" : y_test.values, "pred" : y_pred})

plt.figure(figsize=(10,6))
sns.scatterplot(data = plot_best_model, x="real", y="pred", alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--r', linewidth=2)
plt.title('Valores Reales vs Predicciones (Días de Entrega)', fontsize=14)
plt.xlabel('Días Reales', fontsize=14)
plt.ylabel('Días Predichos', fontsize=14)
plt.show()

**Interpretación del Gráfico (Reales vs. Predicciones):**
- **Rango 0-25 días (entre P90-P95, ver 3.1):** puntos densos sobre la diagonal ($y = \hat{y}$).
- **Cola > 25 días:** puntos bajo la diagonal (subestimación sistemática por pocos ejemplos en colas).


## 6. Importancia de variables (permutación)

Aumento del RMSE al permutar cada feature en test.


In [ ]:
# Permutation importance: cuánto empeora el RMSE (en días) al permutar cada variable
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    best_model, X_test, y_test,
    n_repeats=10, n_jobs=-1, random_state=42,
    scoring="neg_root_mean_squared_error",
)

imp = pd.DataFrame(
    {"Importancia": perm["importances_mean"], "Desvio": perm["importances_std"]},
    index=features,
).sort_values("Importancia")

print(imp.round(3))

plt.figure(figsize=(10, 6))
imp["Importancia"].plot.barh(xerr=imp["Desvio"])
plt.title("Permutation Importance - aumento del RMSE al permutar cada variable")
plt.xlabel("Aumento del RMSE (dias)")
plt.tight_layout()
plt.show()

**Interpretación del Gráfico (Permutation Feature Importance):**
- Aumento del RMSE al permutar cada variable en test.
- **Dominan:** `distance_km` y `estimated_days`.
- **Aporte menor:** `total_volume_cm3`, `total_weight_g`, `is_same_state`, `n_items`, `total_freight`.
- `order_month`: aporte aislado bajo en test.


### 5.2 Residuos vs Predicción

Grafica `e = real - predicho` contra lo predicho.


In [ ]:
# Residuos vs predicción: el "embudo" confirma la heterocedasticidad
resid_df = pd.DataFrame({"pred" : y_pred, "residuo" : y_test.values - y_pred})
plt.figure(figsize=(10, 6))
sns.scatterplot(data=resid_df, x="pred", y="residuo", alpha=0.3)
plt.axhline(0, color="red", linestyle="--", linewidth=2)
plt.title("Residuos vs Predicción (Días de Entrega)")
plt.xlabel("Días Predichos")
plt.ylabel("Residuo (real - predicho)")
plt.show()

**Interpretación del Gráfico (Análisis de Residuos vs. Predicción):**
- Residuos ($e_i = y_{real} - \hat{y}_{pred}$) vs predicho; línea en $0$ = error nulo.
- **Embudo:** la dispersión crece con el valor predicho ($Var(e|\hat{y})$ no constante).
- La transformación $\log(1+y)$ lo atenúa en la masa central sin eliminarlo.


## 7. Diagnóstico global

1. **Masa central (0-25 días):** MAE **3.75 días**, R2 **0.2921**.
2. **Colas:** subestimación y varianza creciente (`Var(e|y_pred)` no constante); intervalos homocedásticos no valen en colas.


## 8. Conclusiones: cómo ayuda este análisis

- **Operación típica:** referencia fiable para entrega 0-25 días (MAE 3.75).
- **Qué mirar ante un desvío:** `distance_km` y `estimated_days`.
- **Pedidos de riesgo:** marcar >25 días en lugar de dar fecha en firme.
